# Rung 40 — THE MERGE GATE

**Does Unsloth's `save_pretrained_merged` carry `modules_to_save` weights into the merged
checkpoint?**

It is a question about a **code path**, not an architecture and not a metric. If the answer is no,
the connector arm would train ~5.9 h on a 27B and ship the **base model** in exactly the two layers
the arm is about — a failure invisible from the score. That is why this is a gate.

Three assertions, each raising (RULES §7 — a gate that fires is a finding):

| | | |
|---|---|---|
| **G1** | coverage | the 2 merger layers are wrapped, and the LoRA legs survive |
| **G2** | it trained | the wrapped weights moved away from base |
| **G3** | 🎯 the merge | the MERGED checkpoint still carries the movement |

**Outcome is pre-declared** (`PLAN.md` §2e): PASS → **path A** (`modules_to_save`, full paths).
FAIL → **path B** (explicit suffix list, never touches the merge). **A FAIL is publishable.**

Proxy: `Qwen3.5-2B` — same class, same connector names, no `deepstack`, like the 27B.
Data: **synthetic noise we generate**. No challenge frames, so the DUA is not engaged.

Run headless:
```
papermill 00_merge_gate.ipynb <out>.ipynb --log-output
```

## 1 — Environment. Fail here, not after loading a model.

In [ ]:
import unsloth  # noqa: F401  MUST precede transformers or the patches are lost
import importlib.metadata as md
import sys, torch

# The gate proves something about a CODE PATH, so the path is asserted, not assumed.
# A PASS under a different unsloth than the arm will run does not transfer.
EXPECT = {"unsloth": "2026.8.15", "unsloth_zoo": "2026.8.10"}
for pkg, want in EXPECT.items():
    got = md.version(pkg)
    assert got == want, (
        f"{pkg} is {got}, expected {want}. This gate would measure a different code "
        "path than the arm runs, so its PASS would not transfer. Re-pin or re-declare."
    )

assert torch.cuda.is_available(), "no CUDA — Unsloth's own documented install has produced a CPU-only torch here before"
print("gpu       ", torch.cuda.get_device_name(0))
print("capability", torch.cuda.get_device_capability(0))
print("free GiB  ", round(torch.cuda.mem_get_info()[0] / 2**30, 1))
for p in ("unsloth", "unsloth_zoo", "peft", "transformers", "torch", "trl"):
    print(f"{p:<14}", md.version(p))

## 2 — Config. Inline, per the repo spec — never edited into the library.

In [ ]:
import logging, sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

TOOLS = str(Path.cwd() / "_tools")
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

from merge_gate import MergeGateConfig, run_gate  # noqa: E402

cfg = MergeGateConfig(
    base_model="Qwen/Qwen3.5-2B",
    hf_home="/data/uaq_user/hf_cache",
    work_dir="/data/uaq_user/tmp/leo_gate40",
    # target_modules MUST stay the bare string: a list containing "all-linear"
    # would not expand (verified in the installed peft 0.20.0).
    target_modules="all-linear",
    modules_to_save=(
        "model.visual.merger.linear_fc1",
        "model.visual.merger.linear_fc2",
    ),
    n_rows=32,
    n_steps=20,
    seed=42,
)
cfg

## 3 — Run. Raises on the first failing assertion; papermill turns that into a non-zero exit.

In [ ]:
result = run_gate(cfg)

## 4 — The verdict, and the branch it selects.

In [ ]:
import json

print("VERDICT:", result["verdict"])
print("PATH   :", result["path"])
print()
print("G1 wrapped        :", result["G1"]["wrapped"])
print("G1 LoRA targets   :", result["G1"]["n_targeted"])
print("G2 sum|delta|     :", result["G2"]["sum_abs_delta_trained"])
print("G3 sum|delta|     :", result["G3"]["sum_abs_delta_merged"])
print("G3 tensors found  :", result["G3"]["n_tensors"])
print("train_loss        :", result["train_loss"])
print("grad_norms        :", result["grad_norms"][:5], "...")
print()
print("result file -> ", Path(cfg.work_dir) / cfg.out_json)